# 03 · Call Summaries
**Assessment task 5.** For every processed call, the LLM on Groq (auto-selected model, e.g. `openai/gpt-oss-120b`; temperature 0, JSON output) writes:
**customer issue**, **agent action**, **outcome / next step**, and a short summary.
The prompt forbids invented details and asks for "Not stated" when information is missing, to keep summaries grounded.

In [ ]:
# ---- Setup: open the project folder (run this cell first) ----
import os, sys
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")                       # results are saved to Drive, so they survive restarts
    PROJECT_DIR = "/content/drive/MyDrive/call-center-ai"  # <- change if your folder is elsewhere
else:
    PROJECT_DIR = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
print("Working in:", os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working in: /content/drive/MyDrive/call-center-ai


In [ ]:
!pip install -q groq pandas
import os; os.kill(os.getpid(), 9)  # restart session after install; then run Setup again and continue

In [ ]:
from src import config
from src.llm import GroqLLM
from src.summarizer import CallSummarizer
from src.pipeline import CallCenterPipeline, load_transcripts, done_ids

pipe = CallCenterPipeline(summarizer=CallSummarizer(GroqLLM()))
done = done_ids(config.SUMMARIES_FILE)
for rec in load_transcripts():
    if rec["call_id"] in done:
        continue
    s = pipe.summarize(rec)
    print(rec["call_id"], "→", s["summary"])

Groq model: openai/gpt-oss-120b (fallbacks: ['openai/gpt-oss-20b', 'qwen/qwen3.8-27b'])
call_001 → The customer reported a delayed package containing important documents. The agent traced the shipment, upgraded it to expedited service for $30, and confirmed delivery by 4:30 pm, after which the customer received a confirmation email.
call_002 → The customer reported a delayed package containing important documents. The agent located the package in a Mississauga sorting center, explained strike-related delays, offered a detailed tracking link, and provided a direct contact number for further assistance. The issue remains unresolved pending delivery.
call_003 → The customer sought help installing a large design program and asked about system capacity, backup, and safe download methods. The agent verified specs, outlined backup and installation steps, promised to email the official link, and offered remote assistance if needed. The customer will proceed with the provided instructions.
call

In [ ]:
import pandas as pd
from src.utils import read_jsonl
summ = pd.DataFrame(read_jsonl(config.SUMMARIES_FILE)).drop_duplicates("call_id", keep="last")
pd.set_option("display.max_colwidth", 200)
summ

,call_id,customer_issue,agent_action,outcome,summary
0,call_001,The customer’s important package did not arrive on time as expected.,"The agent traced the shipment, offered and processed an expedited upgrade for an additional $30, and collected payment details.",resolved,"The customer reported a delayed package containing important documents. The agent traced the shipment, upgraded it to expedited service for $30, and confirmed delivery by 4:30 pm, after which the ..."
1,call_002,"Package with divorce papers is delayed and not yet arrived, needing urgent delivery to Ottawa.","Provided tracking details, explained delay due to strikes, sent a detailed tracking link to the customer's email, and gave a direct contact number for the Ottawa dispatch center.",next step,"The customer reported a delayed package containing important documents. The agent located the package in a Mississauga sorting center, explained strike-related delays, offered a detailed tracking ..."
2,call_003,"Customer needs assistance installing a 4 GB design application on a new Lenovo laptop running Windows 11 and wants guidance on system requirements, backup, and secure download.","Agent confirmed the laptop meets requirements, advised backing up data, provided steps for download and installation, offered to email the official link, explained remote support and warranty cove...",next step,"The customer sought help installing a large design program and asked about system capacity, backup, and safe download methods. The agent verified specs, outlined backup and installation steps, pro..."
3,call_004,Looking for a short‑term (3‑6 months) furnished apartment in the Loughborough area.,"Provided details of an available one‑bedroom furnished flat, explained rent, deposit, parking and security features, discussed early termination fees, and scheduled a viewing for the next day at 1...",next step,"The customer sought a short‑term furnished rental in Loughborough. The agent identified a suitable one‑bedroom flat, covered costs, parking, security and termination terms, and arranged a viewing ..."
4,call_005,"Laptop overheating, especially during Teams video calls.","Provided basic cooling tips, advised bringing the laptop in for diagnosis, scheduled a Friday drop‑off, and promised email confirmation of the appointment and warranty coverage details.",next step,"The customer reports overheating during Teams calls. The agent suggested cooling measures, arranged a diagnostic appointment for Friday, and confirmed warranty coverage for manufacturing defects. ..."
5,call_006,"Received a jacket that is two sizes too small (ordered 3XL, got L).","Guided the customer to upload photos, complete the return form, and arrange a prepaid shipping label for the return.",next step,"The customer reported receiving an incorrectly sized jacket. The agent instructed the customer to submit photos and a return request, then set up a prepaid shipping label. Refund will be processed..."
6,call_007,Wants to open a retirement investment account and receive guidance on portfolio diversification and fund performance,"Initiated the retirement account application, scheduled a branch visit with required documents, and booked a follow‑up appointment with a financial advisor for portfolio diversification",next step,"The customer requested a new retirement investment account and advice on portfolio diversification and recent fund performance. The agent started the application, instructed the customer to visit ..."
7,call_008,"The customer wanted comprehensive details about the Rhythm Beats Live concert, including timing, seating categories and prices, food and parking options, security measures, ticket purchase methods...","The agent supplied all requested information, covering concert schedule, ticket types and costs, venue amenities, security protocols, online and physical ticketing options, and clarified that tick...",resolved,"The agent answered all of the customer's queries about the upcoming concert, 

## Manual grounding check
Read the transcript and summary of 3 calls side by side. Is every fact in the summary present in the transcript?

In [ ]:
from src.utils import format_turns
for rec in load_transcripts()[:3]:
    print("=" * 100, "\n", rec["call_id"])
    print(format_turns(rec["turns"], max_chars=1500))
    print("\nSUMMARY:", summ.set_index("call_id").loc[rec["call_id"]].to_dict(), "\n")

 call_001
Agent: Good morning. Thank you for calling Fox Korea. My name is Ali. How can I help you today?
Customer: Hi, my name is Jenny. I'm waiting for a really important package, but it hasn't has not arrived on time and I was told it would arrive by yesterday still not here I really need this package oh
Agent: I'm sorry to hear dad I understand you're calling about a package that hasn't arrived on time especially since it contains important information let's get this sorted out for you and you wanted this urgently can i please have your tracking number so i can look up the shipment details
Customer: yeah it's four five nine two one seven eight four five
Agent: sorry the last numbers were eight four five was it yeah okay thank you just let me bring this on the system.
Agent: Thank you Jenny. I can see here that your package was due to arrive on 24th.
Agent: Okay but you understand how there was some storms happening in the last couple of days. I understand that it's been delayed due

## Automatic number-grounding check
Every number in each summary is searched for in that call's transcript; numbers not found in the same form are then searched for in other formats (e.g. "4.30", "four").

In [ ]:
# Grounding check: every number in the summary should appear in the transcript
import re
tx = {r["call_id"]: " ".join(t["text"] for t in r["turns"]).lower() for r in load_transcripts()}
for _, row in summ.iterrows():
    text = f"{row['summary']} {row['agent_action']}"
    nums = sorted(set(re.findall(r"\d+(?:[.:]\d+)?", text)))
    missing = [n for n in nums if n not in tx[row["call_id"]]]
    print(f"{row['call_id']}: numbers={nums}  not_in_transcript={missing}")

call_001: numbers=['30', '4:30']  not_in_transcript=['4:30']
call_002: numbers=[]  not_in_transcript=[]
call_003: numbers=[]  not_in_transcript=[]
call_004: numbers=['10']  not_in_transcript=[]
call_005: numbers=[]  not_in_transcript=[]
call_006: numbers=[]  not_in_transcript=[]
call_007: numbers=[]  not_in_transcript=[]
call_008: numbers=[]  not_in_transcript=[]
call_009: numbers=['000', '1', '1000']  not_in_transcript=[]
call_010: numbers=['10']  not_in_transcript=[]
call_011: numbers=['000', '100', '783']  not_in_transcript=[]
call_012: numbers=['3']  not_in_transcript=[]
call_013: numbers=['11']  not_in_transcript=[]
call_014: numbers=['230.45', '4']  not_in_transcript=[]
call_015: numbers=['4', '6']  not_in_transcript=['4']
call_016: numbers=['9:30']  not_in_transcript=['9:30']
call_017: numbers=[]  not_in_transcript=[]
call_018: numbers=['9:30']  not_in_transcript=['9:30']
call_019: numbers=['15', '3', '30', '315', '7', '8']  not_in_transcript=[]
call_020: numbers=['8']  not_in_t

In [ ]:
# Look for the flagged numbers written in other forms, and show the context
checks = {
    "call_001": ["4.30", "4 30", "430", "four thirty", "half past four", "4:30"],
    "call_015": ["four", " 4 ", "4."],
    "call_016": ["9.30", "9 30", "930", "nine thirty", "half past nine"],
    "call_018": ["9.30", "9 30", "930", "nine thirty", "half past nine"],
}
for cid, patterns in checks.items():
    print("=" * 80, f"\n{cid} summary: {summ.set_index('call_id').loc[cid, 'summary']}")
    t = tx[cid]
    found = False
    for p in patterns:
        for m in re.finditer(re.escape(p), t):
            found = True
            print(f"  '{p}': ...{t[max(0, m.start() - 70): m.end() + 70]}...")
    if not found:
        print("  ❌ not found in any form -> possible hallucination")

call_001 summary: The customer reported a delayed package containing important documents. The agent traced the shipment, upgraded it to expedited service for $30, and confirmed delivery by 4:30 pm, after which the customer received a confirmation email.
  '4.30': ...kay, yep. it is confirming that you should be able to receive that by 4.30 p.m. today. that would be perfect. so for that, i will need your trac...
  '4.30': ...l free to call us any time, especially if you didn't receive it up to 4.30pm this afternoon. thank you so much, jenny, and have a great day. you...
call_015 summary: The agent detailed the subsidy program, eligibility, required paperwork, and how to apply online, also mentioning low‑interest loans and a 4‑6 week decision window. The customer was given contact information and will proceed with an online application.
  'four': ...as soon as applications are complete. so i would say there is between four to six weeks wait. okay. and then you'll let me know by email? ye

## Summary validation

**Automatic number-grounding check:** every number in the 20 summaries (prices, times, durations) was searched for in the transcript. 4 numbers were not found in their exact form ("4:30", "9:30", "4"), but all 4 appeared in another format ("4.30 p.m.", "9.30", "four to six weeks"). **Result: 0 hallucinated numbers in 20 summaries.**

**Manual check (3 calls):** summaries match the transcript: issue, agent action and outcome are correct.

**Observations / limitations:**
- The model sometimes paraphrases rather than quoting: call_001 says "important *documents*" while the customer said "important *package / information*". The meaning is close but not word-for-word.
- The `outcome` field is often a short label ("resolved", "next step") rather than a sentence; the full outcome is in the `summary` field.
- The number check catches invented numbers only; it cannot detect invented facts without numbers, which is why a manual review is also done in notebook 05.